In [ ]:
%pip install ipykernel
%pip install opencv-python 
%pip install matplotlib
%pip install matplotlib tk
%pip install plotly nbformat
%pip install scipy
%pip install pyqtgraph PyOpenGL PyQt5
%matplotlib qt


In [ ]:
import time
from collections import deque
from dataclasses import dataclass

import cv2 as cv
import matplotlib
import numpy as np

In [ ]:

def use_window_backend():
    """Notebooks default to an inline backend that closes live figures."""
    try:
        shell = get_ipython()
    except NameError:
        return
    for backend in ("qt", "tk"):
        try:
            shell.run_line_magic("matplotlib", backend)
            return
        except Exception:
            continue


use_window_backend()
import matplotlib.pyplot as plt  # noqa: E402


FRAME_SHAPE = (384, 256, 2)
FRAME_SIZE = int(np.prod(FRAME_SHAPE))


@dataclass
class Config:
    camera_index: int = 0
    left_crop: int = 98
    zoom: int = 4
    rows: int = 80
    column_step: int = 4
    update_seconds: float = 0.1
    figure_size: tuple = (18, 8)      # wide enough for two 3D panels
    temp_min: float = 10.0
    temp_max: float = 50.0
    contrast: float = 0.12
    min_strength: float = 0.2


@dataclass
class Scale:
    low: float
    high: float
    unit: str

    @property
    def span(self):
        return self.high - self.low


def decode_temps(frame):
    """Return a 192x256 array in °C from a raw TC001 frame."""
    data = frame.reshape(FRAME_SHAPE)[FRAME_SHAPE[0] // 2:]
    raw = data[..., 0].astype(np.uint16) + data[..., 1].astype(np.uint16) * 256
    return (raw / 64.0 - 273.15).astype(np.float32)


def decode_gray(frame, crop):
    """Return the cropped brightness image (0 to 255) from a normal colour frame."""
    return cv.cvtColor(frame[:, crop:], cv.COLOR_BGR2GRAY).astype(np.float32)


def open_raw(index):
    """Return a capture that delivers raw TC001 frames, or None if it doesn't."""
    capture = cv.VideoCapture(index)
    capture.set(cv.CAP_PROP_CONVERT_RGB, 0)
    ok, frame = capture.read()
    if ok and frame.size == FRAME_SIZE:
        return capture
    capture.release()
    return None


class Camera:
    def __init__(self, cfg):
        self.crop = cfg.left_crop
        self.capture = open_raw(cfg.camera_index)
        self.raw = self.capture is not None
        if not self.raw:
            self.capture = cv.VideoCapture(cfg.camera_index)
        if not self.capture.isOpened():
            raise RuntimeError("Could not open camera. Try another camera_index.")
        self.scale = Scale(cfg.temp_min, cfg.temp_max, "C") if self.raw else Scale(0, 255, "signal")

    def read(self):
        """Return temperatures in °C (raw mode) or brightness (otherwise), or None."""
        ok, frame = self.capture.read()
        if not ok:
            return None
        return decode_temps(frame) if self.raw else decode_gray(frame, self.crop)

    def release(self):
        self.capture.release()


class CameraView:
    def __init__(self, cfg, scale):
        self.zoom = cfg.zoom
        self.scale = scale

    def show(self, values):
        scaled = (values - self.scale.low) / self.scale.span
        image = cv.applyColorMap(np.uint8(np.clip(scaled, 0, 1) * 255), cv.COLORMAP_INFERNO)
        cv.putText(image, f"max {values.max():.1f} {self.scale.unit}", (4, 14),
                   cv.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1)
        cv.imshow("TC001", cv.resize(image, None, fx=self.zoom, fy=self.zoom))

    @staticmethod
    def quit_requested():
        return cv.waitKey(1) & 0xFF in (ord("q"), 27)


class History:
    """Rolling rows of the hottest value in each image column."""

    def __init__(self, width, cfg):
        self.x = np.arange(0, width, cfg.column_step)
        self.step = cfg.column_step
        self.rows = deque(maxlen=cfg.rows)
        self.pending = None

    def add_frame(self, values):
        row = values.max(axis=0)[::self.step].astype(np.float32)
        self.pending = row if self.pending is None else np.maximum(self.pending, row)

    def commit(self):
        if not self.rows:
            self.rows.extend([self.pending] * self.rows.maxlen)
        else:
            self.rows.append(self.pending)
        self.pending = None

    def as_array(self):
        return np.array(self.rows)


def style_dark_axes(ax, xlim, ylim, zlim, zlabel):
    """Shared dark styling for both 3D panels."""
    ax.set_facecolor("black")
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.set_pane_color((0, 0, 0, 0))
    ax.set(xlim=xlim, ylim=ylim, zlim=zlim)
    ax.set_xlabel("Image X", color="0.7")
    ax.set_ylabel("Time", color="0.7")
    ax.set_zlabel(zlabel, color="0.7")
    ax.tick_params(colors="0.6")
    ax.view_init(45, -70)


class SurfacePanel:
    """Left panel: the full surface of the hottest value per column over time."""

    def __init__(self, ax, history, cfg, scale):
        self.ax = ax
        self.history = history
        self.scale = scale
        self.x, self.y = np.meshgrid(history.x, np.arange(cfg.rows))

    def draw(self):
        self.ax.clear()
        self.ax.plot_surface(
            self.x, self.y, self.history.as_array(),
            cmap="inferno", vmin=self.scale.low, vmax=self.scale.high,
            linewidth=0, antialiased=False,
            rcount=self.x.shape[0], ccount=self.x.shape[1],
        )
        style_dark_axes(
            self.ax,
            xlim=(0, self.x.max()), ylim=(0, self.y.max()),
            zlim=(self.scale.low, self.scale.high),
            zlabel=f"Signal ({self.scale.unit})",
        )
        self.ax.set_title("3D map", color="0.85")


class WispPanel:
    """Right panel: 3D particles showing only what is hotter than normal."""

    def __init__(self, ax, history, cfg, scale):
        self.ax = ax
        self.history = history
        self.cfg = cfg
        self.unit = scale.unit
        self.contrast = cfg.contrast * scale.span
        self.x, self.y = np.meshgrid(history.x, np.arange(cfg.rows))

    def hotspots(self):
        values = self.history.as_array()
        excess = np.clip(values - np.median(values, axis=0), 0, None)
        excess = cv.GaussianBlur(excess, (0, 0), sigmaX=0.8, sigmaY=1.2)
        fade = np.linspace(0.15, 1.0, len(excess))[:, None]
        strength = np.clip(excess / self.contrast, 0, 1) * fade
        return excess, strength

    def draw(self):
        excess, strength = self.hotspots()
        top = max(4 * self.contrast, float(excess.max()))
        cells = np.flatnonzero(strength.ravel() > self.cfg.min_strength).repeat(2)
        x, y, heat, power = (a.ravel()[cells] for a in (self.x, self.y, excess, strength))
        jitter = np.random.uniform(-0.5, 0.5, (3, cells.size))

        colors = plt.get_cmap("inferno")(np.clip(heat / top, 0, 1))
        colors[:, 3] = np.clip(power * 1.2, 0, 1)

        self.ax.clear()
        style_dark_axes(
            self.ax,
            xlim=(0, self.x.max()), ylim=(0, self.y.max()), zlim=(0, top),
            zlabel=f"{self.unit} above normal",
        )
        self.ax.set_title("Wisps", color="0.85")
        if cells.size:
            self.ax.scatter(
                x + jitter[0] * self.history.step,
                y + jitter[1],
                heat + jitter[2] * self.contrast * 0.15,
                c=colors, s=4 + 34 * power, depthshade=False, linewidths=0,
            )


class SideBySide:
    """One window holding the surface map (left) and the wisp map (right)."""

    def __init__(self, history, cfg, scale):
        plt.ion()
        self.fig = plt.figure("Thermal 3D + wisps", figsize=cfg.figure_size, facecolor="black")
        left = self.fig.add_subplot(121, projection="3d")
        right = self.fig.add_subplot(122, projection="3d")
        self.fig.subplots_adjust(left=0.02, right=0.98, wspace=0.05)
        self.panels = [
            SurfacePanel(left, history, cfg, scale),
            WispPanel(right, history, cfg, scale),
        ]

    def draw(self):
        """Redraw both panels. Returns False once the window is closed."""
        for panel in self.panels:
            panel.draw()
        plt.pause(0.001)
        return plt.fignum_exists(self.fig.number)


def run(cfg=Config()):
    if "inline" in matplotlib.get_backend().lower():
        raise RuntimeError("Inline backend closes the figure. Run %matplotlib qt, then restart the kernel.")

    camera = Camera(cfg)
    print("Reading temperatures (°C)." if camera.raw else "Reading brightness only: this camera sends processed frames.")
    view = CameraView(cfg, camera.scale)
    history = maps = None
    last_update = 0.0

    try:
        while True:
            values = camera.read()
            if values is None:
                break

            if history is None:
                history = History(values.shape[1], cfg)
                maps = SideBySide(history, cfg, camera.scale)

            view.show(values)
            history.add_frame(values)

            if time.monotonic() - last_update >= cfg.update_seconds:
                last_update = time.monotonic()
                history.commit()
                if not maps.draw():
                    break

            if view.quit_requested():
                break
    finally:
        camera.release()
        cv.destroyAllWindows()
        plt.close("all")


if __name__ == "__main__":
    run()